---
layout: post
title:  Claude on Amazon Bedrock - part 3
date: 2026-09-18
categories: [AI, Claude, AWS]
toc: true
mermaid: true
maths: true
typora-root-url: /User/ojitha/Github/ojitha.github.io
typora-copy-images-to: ../../blog/assets/images/${filename}
---

{% include video-summary.html
   id=""
   content="" %}

<!--more-->

* TOC
{:toc}

---


## Multi-Turn Conversations with Tools

![AI_Magical_Tool_Loop](/../../../../home/ojitha/Github/blog/assets/images/2026-09-18-BedrockClaude_3/AI_Magical_Tool_Loop.jpg)

The lesson **"Multi-Turn conversations with tools"** focuses on creating a flexible conversational loop that gracefully handles both tool requests and direct answers from Claude.

-   ⚠️ **The Core Problem:** Simple tool integrations often assume that _every_ assistant response requires a tool execution. When a user asks a direct question (e.g., _"What is 1+1?"_), Claude responds without invoking a tool. Naively expecting tool outputs leads to empty or invalid tool-result messages in the conversation history.
-   🚦 **The Solution (`stopReason` Inspection):** Every Bedrock `converse` response includes a `stopReason` metadata attribute:
    
    -   `"tool_use"`: Claude wants to invoke one or more tools. The app must execute the tools, return results in a `user` message block (`toolResult`), and continue the conversation loop.
    -   `"end_turn"`: Claude finished generating its complete answer naturally without requesting further tool calls. The app can display the text and exit the loop.
    -   `"max_tokens"` / `"stop_sequence"`: The model hit generation limits or encountered a stop sequence.

```mermaid
---
config:
  securityLevel: loose
  look: handDrawn
---
sequenceDiagram
    autonumber
    actor User
    participant App as Conversation Loop
    participant Bedrock as AWS Bedrock (Claude Haiku 4.5)
    participant Tool as Pydantic Tool Registry

    User->>App: Input prompt (e.g., "What time is it in Sydney?")
    loop While stopReason == "tool_use"
        App->>Bedrock: converse(messages, toolConfig)
        Bedrock-->>App: Returns assistant message & stopReason
        
        alt stopReason == "tool_use"
            App->>Tool: Validate schema & execute tool
            Tool-->>App: Return toolResult payload
            App->>App: Append assistant message & user toolResult to history
        else stopReason == "end_turn"
            App->>User: Display final assistant response
        end
    end
```

In the pervious post[^1], I've explained the use of the Pydantic[^2] to simplify the tool schema.


In [1]:
import boto3
from typing import Any, Dict, List
from pydantic import BaseModel, Field, ValidationError
from datetime import datetime
from zoneinfo import ZoneInfo

# Initialize AWS Bedrock Runtime client for Sydney region
bedrock = boto3.client("bedrock-runtime", region_name="ap-southeast-2")
MODEL_ID = "au.anthropic.claude-haiku-4-5-20251001-v1:0"

# 📦 1. Define Tool Schemas using Pydantic
class GetDateTimeInput(BaseModel):
    """Get current date and time for a given timezone"""
    timezone: str = Field(
        default="UTC",
        description="Target timezone (e.g., 'Australia/Sydney', 'UTC', 'America/New_York')"
    )

# 🛠️ 2. Tool Implementation Functions
def get_current_datetime(data: GetDateTimeInput) -> Dict[str, str]:
    """Returns current datetime for specified timezone."""
    return {
        "timezone": data.timezone,
        "datetime": datetime.now(ZoneInfo(data.timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")
    }

# Tool Registry mapping names to execution functions and Pydantic models
TOOL_REGISTRY = {
    "get_current_datetime": {
        "func": get_current_datetime,
        "schema": GetDateTimeInput
    }
}

# Convert Pydantic schema to Bedrock toolConfig format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_current_datetime",
                "description": GetDateTimeInput.__doc__ or "",
                "inputSchema": {
                    "json": GetDateTimeInput.model_json_schema()
                }
            }
        }
    ]
}

# 🔄 3. Tool Execution Handler
def process_tool_calls(content_blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Extracts toolUse blocks, validates parameters with Pydantic, and returns toolResult content."""
    tool_results = []
    
    for block in content_blocks:
        if "toolUse" in block:
            tool_use = block["toolUse"]
            tool_use_id = tool_use["toolUseId"]
            tool_name = tool_use["name"]
            raw_input = tool_use.get("input", {})

            if tool_name in TOOL_REGISTRY:
                try:
                    # Validate incoming arguments using Pydantic
                    validated_input = TOOL_REGISTRY[tool_name]["schema"](**raw_input)
                    result_data = TOOL_REGISTRY[tool_name]["func"](validated_input)
                    status = "success"
                except ValidationError as err:
                    result_data = {"error": f"Pydantic Validation Error: {err.errors()}"}
                    status = "error"
            else:
                result_data = {"error": f"Tool '{tool_name}' not found."}
                status = "error"

            tool_results.append({
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"json": result_data}],
                    "status": status
                }
            })
            
    return tool_results

# 💬 4. Dynamic Multi-Turn Conversation Loop
def run_conversation(user_prompt: str) -> List[Dict[str, Any]]:
    """Executes conversation, looping dynamically based on stopReason."""
    messages = [
        {"role": "user", "content": [{"text": user_prompt}]}
    ]

    print(f"\n👤 User: {user_prompt}")

    while True:
        # Call Bedrock Converse API
        response = bedrock.converse(
            modelId=MODEL_ID,
            messages=messages,
            toolConfig=tool_config
        )

        output_message = response["output"]["message"]
        stop_reason = response["stopReason"]

        # Append Claude's assistant message to history
        messages.append(output_message)

        # Print text response if generated
        for block in output_message.get("content", []):
            if "text" in block:
                print(f"🤖 Claude: {block['text']}")

        # Check stop condition
        if stop_reason == "tool_use":
            print("🛠️ Action: Claude requested tool execution. Running tools...")
            tool_results = process_tool_calls(output_message["content"])

            # Send tool results back as a user turn
            messages.append({
                "role": "user",
                "content": tool_results
            })
        else:
            print(f"🚦 Conversation turn complete (stopReason: '{stop_reason}')")
            break

    return messages



Test 1: Triggers tool execution

![Multi-turn-activity-diagram](/../blog/assets/images/2026-09-18-BedrockClaude_3/Multi-turn-activity-diagram.jpg)

In [ ]:
run_conversation("What time is it right now in Sydney?")


👤 User: What time is it right now in Sydney?
🛠️ Action: Claude requested tool execution. Running tools...
🤖 Claude: The current time in Sydney is **8:03 PM (20:03)** on September 18, 2026 (AEST - Australian Eastern Standard Time).
🚦 Conversation turn complete (stopReason: 'end_turn')


[{'role': 'user',
  'content': [{'text': 'What time is it right now in Sydney?'}]},
 {'role': 'assistant',
  'content': [{'toolUse': {'toolUseId': 'tooluse_5GzIh8sFOvyouGjG5ytKGg',
     'name': 'get_current_datetime',
     'input': {'timezone': 'Australia/Sydney'},
     'type': 'tool_use'}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_5GzIh8sFOvyouGjG5ytKGg',
     'content': [{'json': {'timezone': 'Australia/Sydney',
        'datetime': '2026-09-18 20:03:03 AEST'}}],
     'status': 'success'}}]},
 {'role': 'assistant',
  'content': [{'text': 'The current time in Sydney is **8:03 PM (20:03)** on September 18, 2026 (AEST - Australian Eastern Standard Time).'}]}]

Test 2: Direct answer (No tool call required)

In [ ]:
run_conversation("What is the capital of Australia?")


👤 User: What is the capital of Australia?
🤖 Claude: The capital of Australia is **Canberra**. 

Canberra is located in the Australian Capital Territory (ACT) and was purpose-built as the capital city, designed by American architects Walter Burley Griffin and Marion Mahoney Griffin. It was chosen as a compromise location between rivals Sydney and Melbourne and was established in 1927.
🚦 Conversation turn complete (stopReason: 'end_turn')


[{'role': 'user', 'content': [{'text': 'What is the capital of Australia?'}]},
 {'role': 'assistant',
  'content': [{'text': 'The capital of Australia is **Canberra**. \n\nCanberra is located in the Australian Capital Territory (ACT) and was purpose-built as the capital city, designed by American architects Walter Burley Griffin and Marion Mahoney Griffin. It was chosen as a compromise location between rivals Sydney and Melbourne and was established in 1927.'}]}]

<!-- ![Multi-Turn_AI_Tool_Architecture](/../blog/assets/images/2026-09-18-BedrockClaude_3/Multi-Turn_AI_Tool_Architecture.jpg) -->



[^1]: [Claude on Amazon Bedrock - Tool use basics]({% link _posts/2026-09-16-BedrockClaude_2.md%}){:target="_blank" rel="noopener noreferrer"}

[^2]: [Structured Model for AI]({% link _posts/2025-09-06-Python_Type_Annotation.md%}){:target="_blank" rel="noopener noreferrer"}

{:gtxt: .message color="green"}

{:ytxt: .message color="yellow"}

{:rtxt: .message color="red"}
